In [ ]:
# !pip install xgboost

In [ ]:
import pandas as pd
import numpy as np
import json
import joblib
import os

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import ComplementNB
from sklearn.linear_model import SGDClassifier, PassiveAggressiveClassifier
from sklearn.ensemble import ExtraTreesClassifier
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder


from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.metrics import (
    precision_score, recall_score,
    matthews_corrcoef, log_loss
)

In [ ]:
train_df = pd.read_csv("../data/train.csv")
test_df = pd.read_csv("../data/test.csv")

X_train = train_df['headline'].astype(str)
y_train = train_df['category']

X_test = test_df['headline'].astype(str)
y_test = test_df['category']

print(X_train.isna().sum())
print(X_test.isna().sum())

In [ ]:
le = LabelEncoder()
y_train = le.fit_transform(y_train) 
y_test = le.transform(y_test)

In [ ]:
vectorizers = {
    "BoW" : CountVectorizer(max_features=5000),
    "TF-IDF": TfidfVectorizer(max_features=20000, ngram_range=(1,2))
}

In [ ]:
models = {
    "Naive Bayes": MultinomialNB(),
    "ComplementNB": ComplementNB(),
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "Linear SVM_0.5": LinearSVC(C=0.5, class_weight="balanced"),
    "Linear SVM_1.0": LinearSVC(C=1.0, class_weight="balanced"),
    "Linear SVM_2.0": LinearSVC(C=2.0, class_weight="balanced"),
    "SGD": SGDClassifier(loss='hinge', class_weight='balanced', random_state=42),
    "Passive Aggressive": PassiveAggressiveClassifier(random_state=42),
    "Extra Trees": ExtraTreesClassifier(n_estimators=200, random_state=42),
    "XGBoost": XGBClassifier(random_state=42, eval_metric='logloss'),
    "Random Forest": RandomForestClassifier(random_state=42)
}

In [ ]:
X_train.isnull().sum()

In [ ]:
result = []

for vec_name, vectorizer in vectorizers.items():
    
    #transform text
    X_train_vec = vectorizer.fit_transform(X_train)
    X_test_vec = vectorizer.transform(X_test)

    for model_name, model in models.items():

        #train model
        model.fit(X_train_vec, y_train)

        #predict
        y_pred = model.predict(X_test_vec)

        #metrics
        acc = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred, average='weighted')

        print(model_name)
        
        result.append({
            "Vectorizer": vec_name,
            "Model": model_name,
            "Accuracy": round(accuracy_score(y_test, y_pred), 4),
            "F1 Weighted": round(f1_score(y_test, y_pred, average='weighted'), 4),
            "F1 Macro": round(f1_score(y_test, y_pred, average='macro'), 4),      
            "Precision": round(precision_score(y_test, y_pred, average='weighted'), 4),
            "Recall": round(recall_score(y_test, y_pred, average='weighted'), 4),
            "MCC": round(matthews_corrcoef(y_test, y_pred), 4),                  
        })

In [ ]:
results_df = pd.DataFrame(result)

results_df = results_df.sort_values(by="Accuracy", ascending=False)

results_df

In [ ]:
best_row = results_df.iloc[0]

print("Best Combination:")
print(best_row)

In [ ]:
best_vectorizer_name = best_row['Vectorizer']
best_model_name = best_row['Model']

print(best_vectorizer_name)
print(best_model_name)

In [ ]:
best_config = {
    "best_vectorizer": best_vectorizer_name,
    "best_model": best_model_name
}

with open("../models/best_config.json", "w") as f:
    json.dump(best_config, f)

print("Saved:", best_config)

In [ ]:
if best_vectorizer_name == "BoW":
    best_vectorizer = CountVectorizer(max_features=5000)
else:
    best_vectorizer = TfidfVectorizer(max_features=20000, ngram_range=(1,2))


print(best_vectorizer)

In [ ]:
if best_model_name == "Naive Bayes":
    best_model = MultinomialNB()

elif best_model_name == "ComplementNB":
    best_model = ComplementNB()

elif best_model_name == "Logistic Regression":
    best_model = LogisticRegression(max_iter=1000, class_weight="balanced")

elif best_model_name == "Linear SVM_0.5":
    best_model = LinearSVC(C=0.5, class_weight="balanced")

elif best_model_name == "Linear SVM_1.0":
    best_model = LinearSVC(C=1.0, class_weight="balanced")

elif best_model_name == "Linear SVM_2.0":
    best_model = LinearSVC(C=2.0, class_weight="balanced")

else:
    best_model = RandomForestClassifier(random_state=42)

print(best_model)

In [ ]:
X_train_vec = best_vectorizer.fit_transform(X_train)
X_test_vec = best_vectorizer.transform(X_test)

best_model.fit(X_train_vec, y_train)

final_pred = best_model.predict(X_test_vec)

print("FINAL MODEL ACCURACY:", accuracy_score(y_test, final_pred))
print(classification_report(y_test, final_pred))

In [ ]:
os.makedirs("../models", exist_ok=True)

joblib.dump(best_model, "../models/best_model.pkl")
joblib.dump(best_vectorizer, "../models/best_vectorizer.pkl")

print("Model saved successfully!")